In [ ]:
import sys
import os




project_path = r"C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder"
if project_path not in sys.path:
    sys.path.append(project_path)

import track_builder as tb

import pandas as pd
import track_builder as tb


BASE_PATH = r"C:\Users\lamin\Documents\maitrise\ASTD\data"
YEAR      = 2019

MONTHS_TO_LOAD = [1, 2, 3]

USECOLS   = "default"
SAMPLING  = None

COLS_REQUIRED = [
    "shipid",
    "date_time_utc",
    "latitude",
    "longitude",
    "astd_cat",
    "flagname"
]



df = tb.load_astd_monthly(
    BASE_PATH, YEAR, months=MONTHS_TO_LOAD, progress=True,
    usecols=USECOLS, sampling=SAMPLING, remove_nan_rows=COLS_REQUIRED
)





c:\Users\lamin\miniconda3\envs\torch-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading ASTD CSVs: 100%|██████████| 3/3 [03:17<00:00, 65.75s/it]


In [ ]:
df.head()

In [ ]:
tracks = tb.build_ship_tracks(df,
                              max_time_gap_hours=25,
                              max_distance_km=400,
                              min_track_length=1,
                              matching_strategy="balanced",
                              )

Data after cleaning:
  Date range: 2019-01-01 00:00:00+00:00 to 2019-03-31 23:59:58+00:00
  Ship types: ['fishing vessels' 'unknown' 'other activities' 'crude oil tankers'
 'offshore supply ships' 'general cargo ships'
 'other service offshore vessels' 'passenger ships' 'bulk carriers'
 'ro-ro cargo ships' 'cruise ships' 'refrigerated cargo ships'
 'chemical tankers' 'oil product tankers' 'container ships' 'gas tankers']
  Unique ships: 13246
Creating segments for 13246 unique shipids
Created 13246 segments
Sample segment: fishing vessels|iceland|nan|< 1000 gt


C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:240: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = (ship_means.groupby('astd_cat')


In [23]:
track = tracks[tracks["track_id"] == 6131]

track.head()

,month,segment_id,track_id
6491,2019-01,6017,6131
6492,2019-02,13360,6131


In [29]:
# tracks that span at least 2 months
month_counts = tracks.groupby("track_id")["month"].nunique()
multi_month_ids = month_counts[month_counts >= 2].index

print("Number of tracks spanning at least 2 months:", len(multi_month_ids))
tracks[tracks["track_id"].isin(multi_month_ids)].head()


Number of tracks spanning at least 2 months: 449


,month,segment_id,track_id
19,2019-01,1541,20
20,2019-02,1418,20
27,2019-01,171,27
28,2019-02,161,27
29,2019-03,262,27


In [30]:
df = df.copy()
df["month"] = pd.to_datetime(df["date_time_utc"]).dt.strftime("%Y-%m")

df_with_tracks = df.merge(
    tracks,
    left_on=["month", "shipid"],
    right_on=["month", "segment_id"],
    how="inner",
)

fig = tb.plot_ship_tracks(
    df_with_tracks,
    color_by="track_id",
    color_mode="categorical",
    show_points=False,
    map_style="open-street-map",
    title="Tracks for first 3 months of 2019",
)

fig.update_layout(showlegend=False)

#  export to HTML
tb.export_figure(fig, "tracks_3months.html")

In [31]:
# Light version: limit the number of displayed tracks + sampling

import numpy as np
import pandas as pd

# df_tracks must contain positions + track_id for the 3 months
work = df_with_tracks.copy()

print("Initial size:", len(work), "lines")

#  Limit the number of displayed tracks (max 200)
max_tracks = 200
all_ids = work["track_id"].dropna().unique()

if len(all_ids) > max_tracks:
    np.random.seed(42)
    keep_ids = np.random.choice(all_ids, size=max_tracks, replace=False)
    work = work[work["track_id"].isin(keep_ids)]
else:
    keep_ids = all_ids

print(f"Tracks displayed: {len(keep_ids)} / {len(all_ids)}")

#  Sampling: 1 point out of 10 per track
work = (
    work.sort_values(["track_id", "date_time_utc"])
        .groupby("track_id", group_keys=False)
        .apply(lambda g: g.iloc[::10])
)

print("sampling kept :", len(work), "lines")



# Keep only the truly Arctic zone
work = work.query("latitude >= 60 and longitude >= -80 and longitude <= 40")


# delete big jumps in position (e.g., due to track merging)
def remove_big_jumps(g, max_deg=10):
    g = g.sort_values("date_time_utc")
    dlat = g["latitude"].diff().abs()
    dlon = g["longitude"].diff().abs()
    # we keep the first point (diff is NaN) and points where the jump is <= max_deg
    mask = dlat.isna() | ((dlat <= max_deg) & (dlon <= max_deg))
    return g[mask]

work = work.groupby("track_id", group_keys=False).apply(remove_big_jumps)


fig = tb.plot_ship_tracks(
    work,
    color_by="astd_cat",
    color_mode="categorical",
    show_points=True,               
    map_style="open-street-map",
    title="Tracks for first 3 months of 2019 (light version)"
)

fig.update_layout(showlegend=False)

fig.show()

fig.write_html("tracks_3month_light.html", include_plotlyjs='cdn')


Initial size: 15822630 lines
Tracks displayed: 200 / 12735


C:\Users\lamin\AppData\Local\Temp\ipykernel_22280\2575007664.py:28: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

C:\Users\lamin\AppData\Local\Temp\ipykernel_22280\2575007664.py:48: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



sampling kept : 18154 lines
